# A one-loop diagram at local loop-momentum points

This tutorial follows one scalar one-loop graph from generation through its native Feynman-rule expressions and Cross-Free Family (CFF) denominators. We then route two concrete loop momenta through the native momentum basis and evaluate the resulting CFF surfaces. The final numbers probe the local denominator structure; they are not cross sections or loop-integrated amplitudes.

## Generate exactly one loop

The normalized teaching model contains three neutral scalars. We retain only the massless $\phi_0^3$ interaction and request loop order one. Self-loop topologies are generated, but a diagram without a self-edge is selected so its momentum routing is especially easy to inspect.

In [ ]:
import math
from pathlib import Path

import marimo as mo
from symbolica import S
import symbolica.community.feynkit as fk

DATA = next(path for path in (Path("data"), Path("examples/feynkit/data")) if path.exists())
model = fk.Model(DATA / "scalars_2p_3p.json")
options = fk.GenerationOptions(max_vertices=3, allow_self_loops=True)
options.add_vertex_allow(["V_3_SCALAR_000"])
generated = model.generate_diagrams(
    incoming=["scalar_0"],
    outgoing=["scalar_0", "scalar_0"],
    loops=1,
    options=options,
)

diagram = next(
    item
    for item in generated.diagrams
    if all(edge.source != edge.target for edge in item.edges)
)
diagram.validate(model)
basis = diagram.loop_momentum_bases(limit=1)[0]
diagram

## Read the native momentum routing

A `LoopMomentumBasis` expresses every edge momentum in terms of the independent loop momentum $k_0$ and external momenta $p_i$. The coefficients are integers fixed by graph momentum conservation.

In [ ]:
basis

In [ ]:
edges = {edge.id: edge for edge in diagram.edges}
routing_rows = [
    {
        "edge": edge_id,
        "particle": edges[edge_id].particle_name,
        "role": (
            "loop"
            if edge_id in basis.loop_edges
            else "external"
            if edge_id in basis.external_edges
            else "tree"
        ),
        "momentum": signature.format_momentum(),
    }
    for edge_id, signature in basis.edge_signatures.items()
]
routing_rows

## Inspect native Feynman-rule expressions

Internal vertices and propagator edges expose their numerator annotations as native Symbolica `Expression` objects. For this scalar model, each vertex contributes $i\lambda$, every propagator numerator is one, and the diagram numerator is their product, $-i\lambda^3$. The graph-wide factor deliberately keeps `AutG` and the external-fermion ordering sign as symbolic atoms; both evaluate to one for this scalar graph, and the code supplies those values explicitly.

In [ ]:
lam_value = model.parameter("lam").value
rule_point = {S("UFO::lam"): lam_value}
rule_rows = []
for vertex in (item for item in diagram.vertices if not item.is_external):
    rule_expression = vertex.numerator_expression()
    model_rule = model.vertex_rule(vertex.interaction)
    rule_rows.append(
        {
            "object": f"vertex {vertex.id}",
            "model_rule": vertex.interaction,
            "couplings": model_rule.couplings,
            "numerator": mo.as_html(rule_expression.formatted()),
            "value_at_lambda_1": rule_expression.evaluate(rule_point),
        }
    )
for edge in diagram.edges:
    rule_expression = edge.numerator_expression()
    rule_rows.append(
        {
            "object": f"edge {edge.id}",
            "model_rule": f"{edge.particle_name} propagator",
            "couplings": None,
            "numerator": mo.as_html(rule_expression.formatted()),
            "value_at_lambda_1": rule_expression.evaluate(rule_point),
        }
    )
mo.ui.table(rule_rows, selection=None, column_widths={"numerator": 190})

In [ ]:
diagram_numerator = diagram.numerator_expression()
numerator_value = diagram_numerator.evaluate(rule_point)
diagram_numerator.formatted()

In [ ]:
overall_factor = diagram.overall_factor_expression()
overall_value = overall_factor.evaluate(
    {
        S("feynkit_py::AutG")(diagram.symmetry_factor): float(diagram.symmetry_factor),
        S("feynkit_py::ExternalFermionOrderingSign")(1): 1.0,
    }
)
overall_factor.formatted()

## Convert the graph to a Cross-Free Family

CFF rewrites the loop-energy structure as sums of products of causal surfaces. The typed surfaces retain the internal edge energies, signed external-energy shift, and enclosed vertices used to define each formal symbol $E_i$.

In [ ]:
cff = diagram.build_cff(max_orientations=10_000)
cff_expression = cff.to_expression()
report = cff.report
cff

In [ ]:
cff_expression.formatted()

## Choose two nonsingular loop-momentum points

Use arbitrary but consistent energy units. The external massless momenta obey $p_0=p_1+p_2$. We inspect the loop three-vectors $\mathbf{k}_A=(1,2,3)$ and $\mathbf{k}_B=(-2,1,2)$. These are local denominator probes, not a loop integration or phase-space sampling.

In [ ]:
incoming = fk.FourMomentum(10.0, 0.0, 0.0, 10.0)
outgoing_1 = fk.FourMomentum(4.0, 0.0, 0.0, 4.0)
outgoing_2 = fk.FourMomentum(6.0, 0.0, 0.0, 6.0)
external_momenta = [incoming, outgoing_1, outgoing_2]
loop_points = {
    "A": fk.ThreeMomentum(1.0, 2.0, 3.0),
    "B": fk.ThreeMomentum(-2.0, 1.0, 2.0),
}
{
    "external_momenta": external_momenta,
    "loop_points": loop_points,
}

In [ ]:
external_spatial = [
    (momentum.px, momentum.py, momentum.pz) for momentum in external_momenta
]
loop_spatial_by_point = {
    point: [(momentum.px, momentum.py, momentum.pz)]
    for point, momentum in loop_points.items()
}
routed_spatial_by_point = {}
on_shell_energy_by_point = {}
for point, loop_spatial in loop_spatial_by_point.items():
    routed_spatial = {}
    for edge_id, signature in basis.edge_signatures.items():
        routed_spatial[edge_id] = tuple(
            sum(
                coefficient * vector[axis]
                for coefficient, vector in zip(signature.loops, loop_spatial)
            )
            + sum(
                coefficient * vector[axis]
                for coefficient, vector in zip(signature.external, external_spatial)
            )
            for axis in range(3)
        )
    routed_spatial_by_point[point] = routed_spatial
    on_shell_energy_by_point[point] = {
        edge_id: math.sqrt(
            sum(component * component for component in routed_spatial[edge_id])
        )
        for edge_id in basis.tree_edges + basis.loop_edges
    }

external_energy_by_edge = {
    edge_id: external_momenta[index].energy
    for index, edge_id in enumerate(basis.external_edges)
}
[
    {
        "point": point,
        "internal_edge": edge_id,
        "routed_spatial_momentum": tuple(
            round(component, 6) for component in routed_spatial_by_point[point][edge_id]
        ),
        "on_shell_energy": round(energy, 12),
    }
    for point, energies in on_shell_energy_by_point.items()
    for edge_id, energy in sorted(energies.items())
]

## Evaluate the formal CFF expression

FeynKit currently exposes the exact surface definitions but no native momentum-to-surface evaluator or `CffResult.evaluate(...)` method. We therefore use each native `MomentumSignature` above to route the points, then inspect every surface explicitly:

$$E_i=\sum_{e\in +}\omega_e-\sum_{e\in -}\omega_e+\sum_a c_a p_a^0,\qquad \omega_e=\sqrt{\mathbf q_e^2+m_e^2}.$$

This teaching model has massless internal `scalar_0` lines, so the routed on-shell energies are $\omega_e=|\mathbf q_e|$. Once the six surface values are assembled at each point, Symbolica evaluates the native CFF expression directly.

In [ ]:
surface_symbol_values = {}
surface_rows = []
for surface in cff.surfaces:
    value = sum(
        on_shell_energy_by_point["A"][edge] for edge in surface.positive_energies
    ) - sum(
        on_shell_energy_by_point["A"][edge] for edge in surface.negative_energies
    ) + sum(
        coefficient * external_energy_by_edge[edge]
        for edge, coefficient in surface.external_shift
    )
    surface_symbol_values[S(surface.symbol_name)] = value
    surface_rows.append(
        {
            "surface": surface.symbol_name,
            "positive_energies": surface.positive_energies,
            "negative_energies": surface.negative_energies,
            "external_shift": surface.external_shift,
            "value": round(value, 9),
        }
    )
surface_rows

In [ ]:
cff_values = {}
for point, on_shell_energies in on_shell_energy_by_point.items():
    symbol_values = {}
    for surface in cff.surfaces:
        symbol_values[S(surface.symbol_name)] = sum(
            on_shell_energies[edge] for edge in surface.positive_energies
        ) - sum(
            on_shell_energies[edge] for edge in surface.negative_energies
        ) + sum(
            coefficient * external_energy_by_edge[edge]
            for edge, coefficient in surface.external_shift
        )
    cff_values[point] = cff_expression.evaluate(symbol_values)

cff_value = cff_values["A"]
[
    {
        "point": point,
        "loop_momentum": (1.0, 2.0, 3.0) if point == "A" else (-2.0, 1.0, 2.0),
        "local_cff_denominator_probe": value,
    }
    for point, value in cff_values.items()
]

## Interpretation and next steps

The outputs keep the graph bookkeeping factor, native Feynman-rule numerator, and local CFF denominator probes separate. Multiplying those entries would still not create a full integrand: the loop measure, contour or $i0$ prescription, integration, renormalization, and observable normalization are deliberately absent.

For production work, a dedicated evaluator should own the momentum map, particle masses, prescriptions, and numerical stability checks. Until FeynKit exposes that layer natively, the explicit surface table above is the closest auditable local-point inspection supported by the public API.